# Session 3 · Monolith to module

Queen City Trip Analytics, a fictional South End, Charlotte firm, codifies the nightly-trips bucket it created by hand in Session 1.

In [ ]:
source ~/dsba6190-live-demo-03/env.sh && cd "${WORKDIR:?}" && echo "$PROJECT   suffix $SUFFIX" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · The monolith

In [ ]:
cat main.tf

## Step 2 · Read the diff before you cause it

In [ ]:
terraform plan

## Step 3 · Apply, then apply again

In [ ]:
terraform apply -auto-approve

In [ ]:
terraform apply -auto-approve

## Step 4 · Update in place

In [ ]:
cp main.tf.nearline main.tf
terraform plan

In [ ]:
terraform apply -auto-approve

## Step 5 · Destroy and recreate. Stop.

In [ ]:
cp main.tf.renamed main.tf
terraform plan

In [ ]:
cp main.tf.nearline main.tf

## Step 6 · Terraform finds the drift

In [ ]:
# Or make the same edit in the Console
gcloud storage buckets update "gs://${BUCKET:?}" --update-labels=owner=someone-at-2am

In [ ]:
terraform plan

## Step 7 · One module, two environments

In [ ]:
cd "${WORKDIR:?}" && terraform destroy -auto-approve

In [ ]:
cp main.tf.module main.tf
terraform init

In [ ]:
terraform plan

In [ ]:
terraform apply -auto-approve

## Step 8 · Validation, and labels a caller cannot override

In [ ]:
cp -R stages/05-validate/. .
terraform init >/dev/null
cp main.tf.typo main.tf
terraform plan

In [ ]:
cp stages/05-validate/main.tf main.tf
terraform plan

In [ ]:
terraform apply -auto-approve

In [ ]:
terraform output dev_labels

## Step 9 · Safe deletion: prevent_destroy, then force_destroy

In [ ]:
cp -R stages/06-protect/. .
terraform apply -auto-approve

In [ ]:
gcloud storage cp trips-2026-09-02.csv "gs://${PROD_BUCKET:?}/raw/"

In [ ]:
cd "${WORKDIR:?}" && terraform destroy -auto-approve

In [ ]:
cp modules/data-lake/main.tf.unguarded modules/data-lake/main.tf
cd "${WORKDIR:?}" && terraform destroy -auto-approve -target=module.prod_lake

In [ ]:
gcloud storage ls "gs://${PROD_BUCKET:?}/raw/"

## Step 10 · Environments from one map

In [ ]:
cp -R stages/07-foreach/. .
terraform init >/dev/null
terraform plan

In [ ]:
terraform apply -auto-approve

In [ ]:
cp main.tf.test main.tf
terraform plan

In [ ]:
terraform apply -auto-approve

## Step 11 · The trips VM as a module

In [ ]:
cp -R stages/08-vm/. .
terraform init >/dev/null
terraform plan -var machine_type=n2-standard-32

In [ ]:
date +%s > vm.start
terraform apply -auto-approve

In [ ]:
gcloud compute instances delete "${VM:?}" --zone "${ZONE:?}" --quiet

In [ ]:
gcloud compute instances describe "${VM:?}" --zone "${ZONE:?}" \
  --format='table(name,machineType.basename(),status,deletionProtection)'

In [ ]:
terraform apply -auto-approve -var vm_environment=dev

In [ ]:
cp stages/07-foreach/main.tf.test main.tf
terraform apply -auto-approve

In [ ]:
S=$(( $(date +%s) - $(cat vm.start) ))
python3 -c "s=$S; print(f'{s} s, {s/3600:.4f} machine-hours, e2-micro \${s/3600*0.008376:.6f}, n2-standard-32 \${s/3600*1.5539:.4f}')"

## Step 12 · Drift as an exit code, and what makes a run reproducible

In [ ]:
terraform plan -detailed-exitcode > /dev/null; echo "exit code $?"

In [ ]:
# Or make the same edit in the Console
gcloud storage buckets update "gs://${PROD_BUCKET:?}" --update-labels=owner=someone-at-2am

In [ ]:
terraform plan -detailed-exitcode | grep -E "owner|Plan:"; echo "exit code ${PIPESTATUS[0]}"

In [ ]:
sed -n '1,12p' .terraform.lock.hcl

In [ ]:
terraform providers

## Step 13 · Teardown, and proof that nothing remains

In [ ]:
gcloud storage rm "gs://${PROD_BUCKET:?}/**"

In [ ]:
cd "${WORKDIR:?}" && terraform destroy -auto-approve

In [ ]:
gcloud storage buckets list --filter="name~${SUFFIX:?}" --format="value(name)"
gcloud compute instances list --filter="name~${SUFFIX:?}" --format="value(name)"
terraform state list
echo "listed above: nothing means nothing remains"